# Setup — Base de datos de Ventas

Crear el archivo `../data/raw/ventas.db` con las tablas `clientes`, `productos` y `ventas`, incluyendo errores reales inyectados a propósito (negativos, huérfanos, nulos, duplicados).

Una vez generado el `.db`, **cualquier otro notebook** en `notebooks/` puede conectarse a él sin volver a correr este setup.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path

np.random.seed(7)

# Ruta hacia data/raw respetando tu estructura de carpetas
# notebooks/ -> ../data/raw/
DB_PATH = Path("../data/raw/ventas.db")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

conn = sqlite3.connect(DB_PATH)
print(f"Conectado a: {DB_PATH.resolve()}")

## Tabla `clientes`

In [2]:
n_clientes = 60
ciudades = ['Bogotá','Medellín','Cali','Barranquilla','Cartagena','Bucaramanga']
segmentos = ['Premium','Estándar','Básico']

clientes = pd.DataFrame({
    'cliente_id': range(1, n_clientes+1),
    'nombre_cliente': [f'Cliente_{i}' for i in range(1, n_clientes+1)],
    'ciudad': np.random.choice(ciudades, n_clientes),
    'segmento': np.random.choice(segmentos, n_clientes, p=[0.2,0.5,0.3]),
    'fecha_registro': pd.date_range('2022-01-01', periods=n_clientes, freq='15D').strftime('%Y-%m-%d')
})

# Inconsistencias de formato intencionales (premium/Premium/PREMIUM...)
idx_dirty = np.random.choice(n_clientes, 8, replace=False)
clientes.loc[idx_dirty, 'segmento'] = np.random.choice(['premium','PREMIUM','estandar','Estandar'], 8)

clientes.head()

,cliente_id,nombre_cliente,ciudad,segmento,fecha_registro
0,1,Cliente_1,Cartagena,Estándar,2022-01-01
1,2,Cliente_2,Medellín,Estándar,2022-01-16
2,3,Cliente_3,Barranquilla,Básico,2022-01-31
3,4,Cliente_4,Barranquilla,Estándar,2022-02-15
4,5,Cliente_5,Cartagena,Básico,2022-03-02


## Tabla `productos`

In [3]:
n_productos = 25
categorias = ['Electrónica','Hogar','Ropa','Deportes','Belleza']

productos = pd.DataFrame({
    'producto_id': range(1, n_productos+1),
    'nombre_producto': [f'Producto_{i}' for i in range(1, n_productos+1)],
    'categoria': np.random.choice(categorias, n_productos),
    'precio_unitario': np.round(np.random.uniform(10000, 500000, n_productos), -2),
    'costo_unitario': np.round(np.random.uniform(5000, 300000, n_productos), -2)
})

productos.head()

,producto_id,nombre_producto,categoria,precio_unitario,costo_unitario
0,1,Producto_1,Belleza,437600.0,73600.0
1,2,Producto_2,Electrónica,21700.0,138100.0
2,3,Producto_3,Electrónica,143100.0,86500.0
3,4,Producto_4,Belleza,145800.0,153000.0
4,5,Producto_5,Hogar,69100.0,277200.0


## Tabla `ventas` (tabla de hechos)

Incluye errores reales inyectados a propósito:
- Cantidades negativas
- Clientes huérfanos (`cliente_id` que no existe en `clientes`)
- Descuentos nulos
- Filas duplicadas

In [4]:
n_ventas = 400

ventas = pd.DataFrame({
    'venta_id': range(1, n_ventas+1),
    'cliente_id': np.random.randint(1, n_clientes+1, n_ventas),
    'producto_id': np.random.randint(1, n_productos+1, n_ventas),
    'cantidad': np.random.randint(1, 8, n_ventas),
    'fecha_venta': pd.to_datetime(
        np.random.choice(pd.date_range('2024-01-01', '2024-12-31'), n_ventas)
    ).strftime('%Y-%m-%d'),
    'descuento_pct': np.random.choice([0, 5, 10, 15, 20], n_ventas, p=[0.5,0.2,0.15,0.1,0.05])
})

# --- Errores intencionales ---
idx_neg = np.random.choice(n_ventas, 6, replace=False)
ventas.loc[idx_neg, 'cantidad'] = -1

idx_orphan = np.random.choice(n_ventas, 5, replace=False)
ventas.loc[idx_orphan, 'cliente_id'] = 999  # no existe en clientes

idx_null = np.random.choice(n_ventas, 10, replace=False)
ventas.loc[idx_null, 'descuento_pct'] = np.nan

ventas = pd.concat([ventas, ventas.iloc[[10, 50, 200]]], ignore_index=True)  # duplicados

ventas.head()

,venta_id,cliente_id,producto_id,cantidad,fecha_venta,descuento_pct
0,1,23,3,6,2024-09-21,0.0
1,2,50,9,4,2024-01-23,0.0
2,3,47,23,7,2024-11-22,5.0
3,4,10,14,5,2024-11-21,20.0
4,5,36,16,2,2024-12-09,10.0


## Cargar a SQLite (`ventas.db`)

In [ ]:
clientes.to_sql('clientes', conn, index=False, if_exists='replace')
productos.to_sql('productos', conn, index=False, if_exists='replace')
ventas.to_sql('ventas', conn, index=False, if_exists='replace')
conn.commit()

print("="*60)
print("BASE DE DATOS GUARDADA EN DISCO")
print("="*60)
print(f"Archivo:   {DB_PATH.resolve()}")
print(f"clientes:  {len(clientes)} filas")
print(f"productos: {len(productos)} filas")
print(f"ventas:    {len(ventas)} filas (incluye errores intencionales)")

## Verificación rápida

In [6]:
def sql(query, params=None):
    return pd.read_sql_query(query, conn, params=params)

sql("SELECT * FROM ventas LIMIT 5")

,venta_id,cliente_id,producto_id,cantidad,fecha_venta,descuento_pct
0,1,23,3,6,2024-09-21,0.0
1,2,50,9,4,2024-01-23,0.0
2,3,47,23,7,2024-11-22,5.0
3,4,10,14,5,2024-11-21,20.0
4,5,36,16,2,2024-12-09,10.0


In [7]:
conn.close()
print("Conexión cerrada. El archivo ventas.db ya quedó guardado en data/raw/.")

Conexión cerrada. El archivo ventas.db ya quedó guardado en data/raw/.
